In [1]:
from Declare4Py.ProcessModels.DeclareModel import DeclareModel
from Declare4Py.ProcessMiningTasks.Discovery.DeclareMiner import DeclareMiner
from Declare4Py.D4PyEventLog import D4PyEventLog
from Declare4Py.ProcessModels.DeclareModel import DeclareModelTemplate

In [2]:
log_path = r"D:\LTNcoder\.out\eventlogs\paper-0.3-1.xes"

In [3]:
event_log = D4PyEventLog(case_name="case:concept:name")
event_log.parse_xes_log(log_path)

c:\Users\devas\anaconda3\envs\ltn\lib\site-packages\pm4py\util\dt_parsing\parser.py:82: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/5000 [00:00<?, ?it/s]

In [4]:
discovery = DeclareMiner(log=event_log, consider_vacuity=False, min_support=0.05, itemsets_support=0.05, max_declare_cardinality=2)
declare_model: DeclareModel = discovery.run()
print(f"Total constraints discovered: {len(declare_model.serialized_constraints)}")
model_constraints = declare_model.get_decl_model_constraints()
# print("Model constraints:")
# print("-----------------")
# for idx, constr in enumerate(model_constraints):
#     print(idx, constr)


Computing discovery ...
Total constraints discovered: 1538


In [5]:
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareAnalyzer import MPDeclareAnalyzer
from Declare4Py.ProcessMiningTasks.ConformanceChecking.MPDeclareResultsBrowser import MPDeclareResultsBrowser

basic_checker = MPDeclareAnalyzer(log=event_log, declare_model=declare_model, consider_vacuity=False)
conf_check_res: MPDeclareResultsBrowser = basic_checker.run()

In [ ]:
import pickle

# Save the conformance checking results to disk to avoid recalculating
def save_conformance_results(conf_check_res, filename='paper3.1_conformance_results.pkl'):
    with open(filename, 'wb') as f:
        pickle.dump(conf_check_res, f)
    print(f"Conformance checking results saved to {filename}")

# Load the conformance checking results from disk
def load_conformance_results(filename='paper3.1_conformance_results.pkl'):
    try:
        with open(filename, 'rb') as f:
            conf_check_res = pickle.load(f)
        print(f"Conformance checking results loaded from {filename}")
        return conf_check_res
    except FileNotFoundError:
        print(f"File {filename} not found. Run conformance checking first.")
        return None

# Save the current conformance results
save_conformance_results(conf_check_res)

Conformance checking results saved to paper3.1_conformance_results.pkl


In [13]:
conf_check_res = load_conformance_results()


Conformance checking results loaded from paper3.1_conformance_results.pkl


In [6]:
conf_check_df =  conf_check_res.get_metric(metric="state")
# display(conf_check_df)

In [7]:
summary_df = conf_check_df.apply(lambda col: col.value_counts()).fillna(0).astype(int)
summary_df = summary_df.reindex([0, 1])
summary_df = summary_df / len(conf_check_df)
summary_df = summary_df.T
summary_df = summary_df.sort_values(by=1, ascending=False)
display(summary_df)

,0,1
Existence1[Final Decision] | |,0.0000,1.0000
"Choice[Conclude, Final Decision] | |",0.0000,1.0000
"Choice[Final Decision, Research Related Work] | |",0.0000,1.0000
"Choice[Review, Experiment] | |",0.0000,1.0000
"Choice[Experiment, Review] | |",0.0000,1.0000
...,...,...
"Response[Develop Method, Evaluate] | |",0.9300,0.0700
"Alternate Response[Develop Method, Evaluate] | |",0.9300,0.0700
"Not Chain Response[Conclude, Submit] | |",0.9440,0.0560
"Not Chain Precedence[Conclude, Submit] | |",0.9446,0.0554


In [8]:
# conf_check_df.value_counts("End[Activity B] | |")

In [9]:
import pandas as pd

In [10]:
activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)
satisfied_df = conf_check_res.get_metric(metric="state").fillna(0)
import pandas as pd
import numpy as np

# Support = how often constraint is satisfied
support = satisfied_df.sum(axis=0) / len(satisfied_df)

# Activation rate = how often constraint is activated
activated_counts = activated_df.sum(axis=0)
activation_rate = activated_counts / len(activated_df)

# Satisfied counts
satisfied_counts = satisfied_df.sum(axis=0)

# Confidence = P(satisfied | activated), safe division
confidence = np.where(
    activated_counts != 0,
    satisfied_counts / activated_counts,
    0
)
confidence = pd.Series(confidence, index=activated_counts.index)

# Combine into a DataFrame
metrics_df = pd.DataFrame({
    'support': support,
    'confidence': confidence,
    'activation_rate': activation_rate
})

C:\Users\devas\AppData\Local\Temp\ipykernel_19588\3686773640.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  activated_df = conf_check_res.get_metric(metric="num_activations").fillna(0)


In [11]:
metrics_df

,support,confidence,activation_rate
Existence1[Final Decision] | |,1.0000,0.000000,0.0000
Absence2[Final Decision] | |,0.9930,0.000000,0.0000
Exactly1[Final Decision] | |,0.9930,0.000000,0.0000
End[Final Decision] | |,0.9912,0.000000,0.0000
Existence1[Identify Problem] | |,0.9950,0.000000,0.0000
...,...,...,...
"Not Precedence[Develop Hypothesis, Conduct Study] | |",0.0930,0.186747,0.4980
"Not Chain Response[Conduct Study, Develop Hypothesis] | |",0.4876,0.979116,0.4980
"Not Chain Response[Develop Hypothesis, Conduct Study] | |",0.8374,0.977814,0.8564
"Not Chain Precedence[Conduct Study, Develop Hypothesis] | |",0.8388,0.979449,0.8564
